In [ ]:
import numpy as np 
import matplotlib.pyplot as plt  
import pandas as pd  
import seaborn as sns
import keras
import tensorflow as tf

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import RMSprop,Adam
from tensorflow.keras.utils import to_categorical

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#importing datasets  
df= pd.read_csv('./drive/MyDrive/Dataset/diabetes.csv')  

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
x=df.iloc[:,0:8]
y=df.iloc[:,-1]

In [ ]:
x[300:310]

In [ ]:
x.shape

In [ ]:
y.shape

In [ ]:
y[100:110]

In [ ]:
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(x)
norm_x = scaler.transform(x)
x_train,x_test,y_train,y_test=train_test_split(norm_x,y,test_size=0.20,random_state=10)

In [ ]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

In [ ]:
y_train[:10]

In [ ]:
y_train=to_categorical(y_train,2)
y_test=to_categorical(y_test,2)

In [ ]:
y_train[:10]

In [ ]:
print(x_train.shape)
print(y_train.shape)

In [ ]:
model=Sequential()
layer1=Dense(32,activation='relu',input_shape=(8,),name="layer1")
model.add(layer1)
model.add(Dropout(0.2))
layer2=Dense(16,activation='relu',name="layer2")
model.add(layer2)
model.add(Dropout(0.2))
layer3=Dense(8,activation='relu',name="layer3")
model.add(layer3)
model.add(Dropout(0.2))
model.add(Dense(2,activation='softmax'))
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

**Early Stopping Regularization**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=2)


In [ ]:
history=model.fit(x_train,y_train,shuffle='True',validation_data=(x_test,y_test),batch_size=10,epochs=100,callbacks=[early_stopping])

**Getting Intermediate Values and weights**

In [ ]:
layer1.get_weights

In [ ]:
layer1.get_input_at

**Model Saving**

In [ ]:
model.save('./drive/MyDrive/Dataset/NNDiabetes', save_format='h5')

# **Model Loading**

In [ ]:
from tensorflow.keras.models import load_model
# returns a compiled model
# identical to the previous one
model123 = load_model('./drive/MyDrive/Dataset/NNDiabetes')

In [ ]:
model123.summary()

In [ ]:
tf.keras.utils.plot_model(model, to_file="./drive/MyDrive/Dataset/diabetesmodel.png", show_shapes=True)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
loss=history.history['loss']
epoch=range(1,len(loss)+1)
plt.plot(epoch,history.history['accuracy'],label='training')
plt.plot(epoch,history.history['val_accuracy'],label='validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.plot(epoch,history.history['loss'],label='training')
plt.plot(epoch,history.history['val_loss'],label='validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
score=model.evaluate(x_test,y_test,verbose=0)
print('total loss',score[0])
print('test accuracy',score[1])

In [ ]:
score

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
y_pred=model.predict(x_test)
pred=np.argmax(y_pred,axis=1)[:5]
label=np.argmax(y_test,axis=1)[:5]
print("Label of the first 5  prediction ",pred)
print("label of the first 5  original ",label)

In [ ]:
y_testnew=np.argmax(y_test,axis=1)
y_prednew=np.argmax(y_pred,axis=1)

In [ ]:
cf_matrix = confusion_matrix(y_testnew,y_prednew)
cf_matrix

In [ ]:
heatmap = sns.heatmap(cf_matrix, annot=True, fmt="d")
plt.xlabel('Predicted',fontsize=20)
plt.ylabel('Actual',fontsize=20)
plt.title('Accuracy Score   {0}'.format(score),fontsize=20)
plt.show()


In [ ]:
history.history.keys()

# **FEATURE EXTRACTION and learning by tracking each layer**

In [ ]:
x_test[0]

In [ ]:
output1=model.get_layer("layer1").output 
intermediate_model=tf.keras.models.Model(inputs=model.input,outputs=output1)
intermediate_prediction=intermediate_model.predict(x_test[0].reshape(1,8))
print(intermediate_prediction.shape)
intermediate_prediction

In [ ]:
output2=model.get_layer("layer2").output 
intermediate_model=tf.keras.models.Model(inputs=model.input,outputs=output2)
intermediate_prediction=intermediate_model.predict(x_test[0].reshape(1,8))
print(intermediate_prediction.shape)
intermediate_prediction

In [ ]:
output3=model.get_layer("layer3").output 
intermediate_model=tf.keras.models.Model(inputs=model.input,outputs=output3)
intermediate_prediction=intermediate_model.predict(x_test[0].reshape(1,8))
print(intermediate_prediction.shape)
intermediate_prediction

# **Transfer learning with a Sequential model**

Transfer learning consists of freezing the bottom layers in a model and only training the top layers. If you aren't familiar with it, make sure to read our guide to transfer learning.

Here are two common transfer learning blueprint involving Sequential models.

First, let's say that you have a Sequential model, and you want to freeze all layers except the last one. In this case, you would simply iterate over model.layers and set layer.trainable = False on each layer, except the last one. Like this:

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(784)),
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(10),
])

# Presumably you would want to first load pre-trained weights.
model.load_weights(...)

# Freeze all layers except the last one.
for layer in model.layers[:-1]:
  layer.trainable = False

# Recompile and train (this will only update the weights of the last layer).
model.compile(...)
model.fit(...)